## Importing Libraries and Definitions

In [1]:
import cv2
import mediapipe as mp
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
Holistic = mp.solutions.holistic.Holistic
def extract_landmarks(landmarks):
    return np.array([[l.x, l.y] for l in landmarks],dtype=np.float32)

### Saving Images by Clicking 

In [14]:
import cv2

cam = cv2.VideoCapture(0)

if not cam.isOpened():
    print("Error: Camera not opened")
    exit()

cv2.namedWindow("Camera")

counter = 0
current_frame = None

def mouse_click(event, x, y, flags, param):
    global counter, current_frame
    if event == cv2.EVENT_LBUTTONDOWN:
        counter += 1
        filename = f"data/scissors/scissors{counter}.png"
        cv2.imwrite(filename, current_frame)
        print(f"Saved {filename}")

cv2.setMouseCallback("Camera", mouse_click)

while True:
    ret, frame = cam.read()
    if not ret:
        print("Failed to grab frame")
        break

    current_frame = frame.copy()
    # current_frame = cv2.flip(current_frame, 1)
    # rgb_frame = cv2.cvtColor(current_frame, cv2.COLOR_BGR2RGB)
    # results = holistic.process(rgb_frame)
        # Make predictions
        
    cv2.imshow("Camera", frame)
    # drawing_utils.draw_landmarks(
    #         frame,
    #         results.right_hand_landmarks,
    #         mp.solutions.holistic.HAND_CONNECTIONS,
    #         connection_drawing_spec=drawing_styles.get_default_hand_connections_style()
    #     )
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cam.release()
cv2.destroyAllWindows()


Saved data/scissors/scissors1.png
Saved data/scissors/scissors2.png


### Pose Processing of Image (irrelevant)

In [15]:
import os
import pickle
dataset = []
target = []
folder_path_paper = "data/paper/" 
folder_path = folder_path_paper
all_files = os.listdir(folder_path)  # lists files and folders
files_only = [f for f in all_files if os.path.isfile(os.path.join(folder_path, f))]
counter = 0
L = len(files_only)
model = Holistic(min_detection_confidence=0.5, min_tracking_confidence=0.5)

for f in files_only:
    counter += 1
    print(100*counter/L,end="\r")
    image = plt.imread(folder_path+f)
    img_model = (image[..., :3] * 255).astype(np.uint8)
    model = Holistic(min_detection_confidence=0.5, min_tracking_confidence=0.5)
    results = model.process(img_model)
    if results.pose_landmarks is not None:
        ldk = extract_landmarks(results.pose_landmarks.landmark)
        ldk_temp = extract_landmarks(results.pose_landmarks.landmark).flatten()
        dataset.append(ldk_temp)

with open(folder_path+'dataset_paper.pkl', 'wb') as file:  # 'wb' = write binary
    pickle.dump(dataset, file)

UnidentifiedImageError: cannot identify image file 'data/paper/dataset_paper.pkl'

### Right-hand Processing of Image (irrelevant)

In [16]:
import os
import pickle
dataset = []
target = []
gestures = ["rock","paper","scissors"]
folder_path_rock = "data/rock/" 
folder_path_scissors = "data/scissors/" 
folder_paths = [folder_path_rock,folder_path_scissors]
# folder_path = folder_path_paper
for gesture in gestures:
    folder_path = "data/"+gesture+"/"
    all_files = os.listdir(folder_path)  # lists files and folders
    files_only = [f for f in all_files 
              if os.path.isfile(os.path.join(folder_path, f)) and f.lower().endswith('.png')]

    counter = 0
    L = len(files_only)
    model = Holistic(min_detection_confidence=0.5, min_tracking_confidence=0.5)

    for f in files_only:
        counter += 1
        print(100*counter/L,end="\r")
        image = plt.imread(folder_path+f)
        img_model = (image[..., :3] * 255).astype(np.uint8)
        model = Holistic(min_detection_confidence=0.5, min_tracking_confidence=0.5)
        results = model.process(img_model)
        if results.right_hand_landmarks is not None:
            ldk_temp = extract_landmarks(results.right_hand_landmarks.landmark).flatten()
            dataset.append(ldk_temp)

    with open(folder_path+'dataset_RightHand_'+gesture+'.pkl', 'wb') as file:  # 'wb' = write binary
        pickle.dump(dataset, file)

### Loading Individual Datasets

In [31]:

import pickle
import numpy as np 
with open("data/paper/dataset_RightHand_paper.pkl", "rb") as f:
     paper = pickle.load(f)
with open("data/rock/dataset_RightHand_rock.pkl", "rb") as f:
     rock = pickle.load(f)
with open("data/scissors/dataset_RightHand_scissors.pkl", "rb") as f:
     scissors = pickle.load(f)

with open("data/paper/Manual_dataset_RightHand_paper.pkl", "rb") as f:
    paper = pickle.load(f)
with open("data/rock/Manual_dataset_RightHand_rock.pkl", "rb") as f:
    rock = pickle.load(f)
with open("data/scissors/Manual_dataset_RightHand_scissors.pkl", "rb") as f:
    scissors = pickle.load(f)

dataset = np.concatenate((np.array(rock),np.array(paper),np.array(scissors)),axis=0)
target = ['rock']*len(rock)+['paper']*len(paper)+['scissors']*len(scissors)
columns = []
for col in range(int(dataset.shape[1]/2)):
    columns.append(f"x{col}")
    columns.append(f"y{col}")

## Loading Merged Datasets

In [40]:
import pickle
with open("data/paper/Manual_dataset_RightHand_paperm.pkl", "rb") as f:
    paper = pickle.load(f)
with open("data/rock/Manual_dataset_RightHand_rockm.pkl", "rb") as f:
    rock = pickle.load(f)
with open("data/scissors/Manual_dataset_RightHand_scissorsm.pkl", "rb") as f:
    scissors = pickle.load(f)

dataset = np.concatenate((np.array(rock),np.array(paper),np.array(scissors)),axis=0)
target = ['rock']*len(rock)+['paper']*len(paper)+['scissors']*len(scissors)
columns = []
for col in range(int(dataset.shape[1]/2)):
    columns.append(f"x{col}")
    columns.append(f"y{col}")


### Converting to DataFrame and Saving to Pickle

In [41]:
import pandas as pd
data = pd.DataFrame(data= dataset,columns=columns,index = range(len(dataset)))
data["target"] = target 
data.to_pickle("Manual_dataset_total_RightHand_Merged")
data

,x0,y0,x1,y1,x2,y2,x3,y3,x4,y4,...,y16,x17,y17,x18,y18,x19,y19,x20,y20,target
0,0.618734,0.678595,0.604782,0.609769,0.615376,0.554255,0.643426,0.533819,0.669728,0.535176,...,0.566734,0.680175,0.570719,0.679044,0.566891,0.670895,0.583055,0.663520,0.593625,rock
1,0.626671,0.707446,0.659186,0.673067,0.667507,0.620117,0.654802,0.577309,0.633523,0.560595,...,0.542434,0.611099,0.548657,0.604627,0.506228,0.600010,0.520371,0.599258,0.538966,rock
2,0.648071,0.692400,0.686409,0.629165,0.696529,0.578373,0.704166,0.543677,0.698406,0.514967,...,0.584879,0.635555,0.559044,0.689876,0.559579,0.690221,0.588521,0.675656,0.598284,rock
3,0.643369,0.699669,0.689657,0.633668,0.702477,0.584984,0.711667,0.551699,0.713211,0.518931,...,0.593843,0.638307,0.557358,0.693700,0.564687,0.692887,0.594612,0.678098,0.603736,rock
4,0.631114,0.599393,0.683996,0.586744,0.721610,0.556735,0.748154,0.528025,0.755447,0.491604,...,0.529272,0.669816,0.483093,0.709637,0.519847,0.697244,0.541506,0.681064,0.537822,rock
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
728,0.259314,0.473709,0.287096,0.459361,0.317989,0.467753,0.337916,0.495281,0.346518,0.509074,...,0.517607,0.297736,0.505227,0.315704,0.544620,0.306422,0.543413,0.299403,0.528641,scissors
729,0.229199,0.491574,0.255494,0.446612,0.291970,0.440271,0.319766,0.465894,0.332529,0.492282,...,0.490153,0.302396,0.530038,0.326998,0.525828,0.318037,0.517627,0.305945,0.515648,scissors
730,0.289635,0.479251,0.296230,0.445721,0.318817,0.425673,0.343345,0.431317,0.357246,0.443127,...,0.479890,0.337999,0.493829,0.360837,0.498023,0.350536,0.503004,0.339304,0.501559,scissors
731,0.369853,0.516866,0.353173,0.473304,0.349881,0.432642,0.360319,0.406518,0.377271,0.394871,...,0.431565,0.370266,0.408361,0.405075,0.403143,0.415437,0.424720,0.419609,0.442284,scissors


## Neural Network Model (93% accuracy)

### Building Model

In [46]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense,Input,Dropout

landmarks_of_interest = range(int((data.shape[1]-1)/2))
length_inputs = len(landmarks_of_interest)
# Create the model
model = Sequential([
    Input((2*length_inputs,)),
    Dense(128,activation='relu'),
    Dropout(0.2),
    Dense(64, activation='relu'),
    Dropout(0.2),
    Dense(32, activation='relu'),
    Dropout(0.2),
    Dense(3, activation='softmax')  # Output layer for 3 classes
])

# Compile the model
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',  # Use for one-hot encoded labels
    metrics=['accuracy']
)

# Model summary
model.summary()


Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_12 (Dense)                │ (None, 128)            │         5,504 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_9 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_13 (Dense)                │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_10 (Dropout)            │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_14 (Dense)                │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_11 (Dropout)            │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_15 (Dense)                │ (None, 3)              │            99 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 15,939 (62.26 KB)

 Trainable params: 15,939 (62.26 KB)

 Non-trainable params: 0 (0.00 B)

### Loading & Splitting into Train/Test

In [43]:
df = pd.read_pickle("Manual_dataset_total_RightHand_Merged")
for index, row in df.iterrows():
    temp = row.copy()               # important!
    temp[0:-1:2] -=  temp.iloc[0:-1:2].min()
    temp[1:-1:2] -=  temp.iloc[1:-1:2].min()
    df.loc[index] = temp
from sklearn.model_selection import train_test_split
X = df[df.columns[:-1]].values
y = df[df.columns[-1]].values
dummies = pd.get_dummies(y).astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, dummies, test_size=0.15, random_state=42)


C:\Users\yousr\AppData\Local\Temp\ipykernel_16236\2690869659.py:6: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.2660316377878189' has dtype incompatible with float32, please explicitly cast to a compatible dtype first.
  df.loc[index] = temp
C:\Users\yousr\AppData\Local\Temp\ipykernel_16236\2690869659.py:6: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.2725473791360855' has dtype incompatible with float32, please explicitly cast to a compatible dtype first.
  df.loc[index] = temp
C:\Users\yousr\AppData\Local\Temp\ipykernel_16236\2690869659.py:6: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.41982485353946686' has dtype incompatible with float32, please explicitly cast to a compatible dtype first.
  df.loc[index] = temp
C:\Users\yousr\

### Training NN Model

In [47]:
from tensorflow.keras.callbacks import EarlyStopping
early_stop = EarlyStopping(
    monitor='val_loss',      # Metric to monitor
    patience=3,              # Stop after 5 epochs of no improvement
    verbose=1,               # Print messages when stopping
    restore_best_weights=True  # Restore model weights from the epoch with the best value of monitored metric
)
history = model.fit(
    X_train,
    y_train.values,
    epochs=300,
    batch_size=32,
    validation_data=(X_test, y_test),
    callbacks = [early_stop]
)

Epoch 1/300
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.4157 - loss: 1.0724 - val_accuracy: 0.4636 - val_loss: 1.0344
Epoch 2/300
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.4591 - loss: 1.0330 - val_accuracy: 0.4636 - val_loss: 1.0064
Epoch 3/300
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.4639 - loss: 0.9938 - val_accuracy: 0.4636 - val_loss: 0.9684
Epoch 4/300
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.4944 - loss: 0.9425 - val_accuracy: 0.5636 - val_loss: 0.9161
Epoch 5/300
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6003 - loss: 0.8892 - val_accuracy: 0.6545 - val_loss: 0.8579
Epoch 6/300
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6774 - loss: 0.8011 - val_accuracy: 0.7273 - val_loss: 0.7832
Epoch 7/300
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7287 - loss: 0.7050 - val_accuracy: 0.8000 - val_loss: 0.6362
Epoch 8/300
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7817 - loss: 0.6233 - val_accuracy: 0.8273 - 

### Saving Model to .pkl

In [48]:
with open('model_new.pkl', 'wb') as file:  # 'wb' = write binary
       pickle.dump(model, file)

## RandomForest Model (66% Accuracy)

In [25]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier()
params = {'n_estimators': range(100,800,100),
          'max_depth' : range(2,5,2)}
cv = RandomizedSearchCV(rf,params,verbose=2,scoring="accuracy",n_jobs=-1)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.15, random_state=42)
history = cv.fit(X_train,y_train)


Fitting 5 folds for each of 10 candidates, totalling 50 fits


In [26]:
pd.DataFrame(history.cv_results_)


,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_n_estimators,param_max_depth,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
0,1.053045,0.280603,0.052186,0.012326,400,2,"{'n_estimators': 400, 'max_depth': 2}",0.500000,0.493333,0.466667,0.493333,0.400000,0.470667,0.037142,6
1,1.667920,0.348799,0.070310,0.007366,700,2,"{'n_estimators': 700, 'max_depth': 2}",0.473684,0.506667,0.466667,0.493333,0.413333,0.470737,0.032006,5
2,0.330147,0.075534,0.024290,0.026084,100,4,"{'n_estimators': 100, 'max_depth': 4}",0.368421,0.333333,0.320000,0.346667,0.333333,0.340351,0.016374,10
3,0.939463,0.192628,0.040487,0.012381,300,4,"{'n_estimators': 300, 'max_depth': 4}",0.368421,0.373333,0.360000,0.293333,0.346667,0.348351,0.028957,8
4,0.272143,0.077768,0.026765,0.017158,100,2,"{'n_estimators': 100, 'max_depth': 2}",0.500000,0.506667,0.466667,0.493333,0.400000,0.473333,0.039101,3
5,0.908124,0.229184,0.043114,0.011510,300,2,"{'n_estimators': 300, 'max_depth': 2}",0.486842,0.506667,0.466667,0.493333,0.426667,0.476035,0.027854,1
6,1.061794,0.020676,0.051259,0.004843,500,2,"{'n_estimators': 500, 'max_depth': 2}",0.500000,0.506667,0.466667,0.493333,0.413333,0.476000,0.034150,2
7,0.460044,0.086095,0.025055,0.006600,200,2,"{'n_estimators': 200, 'max_depth': 2}",0.500000,0.480000,0.466667,0.506667,0.413333,0.473333,0.033200,3
8,1.271899,0.038034,0.036518,0.003781,700,4,"{'n_estimators': 700, 'max_depth': 4}",0.355263,0.333333,0.360000,0.293333,0.373333,0.343053,0.027999,9
9,0.464212,0.029685,0.019132,0.005862,200,4,"{'n_estimators': 200, 'max_depth': 4}",0.394737,0.333333,0.360000,0.360000,0.360000,0.361614,0.019518,7


## Training XgBoost Model (66% Accuracy)

In [27]:
ybis = y
ybis[ybis=='rock'] = 0 
ybis[ybis=='paper'] = 1 
ybis[ybis=='scissors'] = 2 
X_train, X_test, y_train, y_test = train_test_split(
    X, ybis, test_size=0.15, random_state=42)

In [29]:
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score


model = xgb.XGBClassifier(
    objective='multi:softprob',  # probabilities for each class
    num_class=3,
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='mlogloss',
    random_state=42
)
model.fit(
    X_train,
    y_train,
    eval_set=[(X_test, y_test)],
    verbose=True
)


ModuleNotFoundError: No module named 'xgboost'

In [11]:
from sklearn.metrics import accuracy_score
y_pred = model.predict(X_test)
acc = accuracy_score(y_test.astype(int), y_pred)
print(f"Validation Accuracy: {acc:.4f}")


NameError: name 'model' is not defined

In [12]:
import pickle
with open("model.pkl", "rb") as f:
        model = pickle.load(f)


## Live Test with Webcam

In [28]:
import cv2
import mediapipe as mp
import pickle
import cv2
import mediapipe as mp
import pickle
import numpy as np   # <--- add this

def extract_landmarks(landmarks):
    return np.array([[l.x, l.y] for l in landmarks],dtype=np.float32)
mp_holistic = mp.solutions.holistic
mp_drawing = mp.solutions.drawing_utils
drawing_utils = mp.solutions.drawing_utils
drawing_styles = mp.solutions.drawing_styles
gesture = ['paper', 'rock', 'scissors','None']
threshold = .8
cap = cv2.VideoCapture(0)  # Open webcam

if not cap.isOpened():
    print("Error: Camera not opened")
    exit()

cv2.namedWindow("Camera")
with open("model.pkl", "rb") as f:
    model = pickle.load(f)  


   

ind = 3 
with mp_holistic.Holistic(min_detection_confidence=0.5, min_tracking_confidence=0.5) as holistic:
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        # Flip for a mirror view
        # frame = cv2.flip(frame, 1)
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        # Make predictions
        results = holistic.process(rgb_frame)
        if results.right_hand_landmarks is not None:
            features = extract_landmarks(results.right_hand_landmarks.landmark).reshape(1, -1)
            sample = features[:, :]   # already (1, n_features)
            sample[:, 0::2] -= np.min(sample[:, 0::2], axis=1, keepdims=True)
            sample[:, 1::2] -= np.min(sample[:, 1::2], axis=1, keepdims=True)
            # Predict using the pre-trained ML model
            prediction = model.predict(sample,verbose=0)
            print(prediction)
            if (prediction>threshold).any():
                ind = np.argmax(prediction)
                print(prediction)
            else:
                ind = 3
         
        drawing_utils.draw_landmarks(
            frame,
            results.left_hand_landmarks,
            mp.solutions.holistic.HAND_CONNECTIONS,
            connection_drawing_spec=drawing_styles.get_default_hand_connections_style()
        )

        drawing_utils.draw_landmarks(
            frame,
            results.right_hand_landmarks,
            mp.solutions.holistic.HAND_CONNECTIONS,
            connection_drawing_spec=drawing_styles.get_default_hand_connections_style()
        )
        # # Display prediction
        cv2.putText(frame, f'Gesture Detected: {gesture[ind]}', (10, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

        cv2.imshow("Camera", frame)

        if cv2.waitKey(1) & 0xFF == 27:  # ESC to quit
            break

cap.release()
cv2.destroyAllWindows()


[[0.16878752 0.01963325 0.8115792 ]]
[[0.16878752 0.01963325 0.8115792 ]]
[[0.19887172 0.04674792 0.7543804 ]]
[[0.19594279 0.05160078 0.7524564 ]]
[[0.18139723 0.03919648 0.7794062 ]]
[[0.17230387 0.03711984 0.7905763 ]]
[[0.15908055 0.02851655 0.81240284]]
[[0.15908055 0.02851655 0.81240284]]
[[0.16165452 0.029169   0.8091765 ]]
[[0.16165452 0.029169   0.8091765 ]]
[[0.17644766 0.03844345 0.78510886]]
[[0.18167739 0.04183336 0.7764893 ]]
[[0.18568704 0.04475266 0.7695603 ]]
[[0.19043316 0.04695147 0.76261544]]
[[0.18976332 0.04604443 0.7641923 ]]
[[0.19261344 0.04960474 0.7577818 ]]
[[0.19376075 0.04969232 0.75654685]]
[[0.19275092 0.04903143 0.75821775]]
[[0.19215283 0.04863031 0.75921685]]
[[0.19314766 0.04770966 0.7591427 ]]
[[0.19531794 0.04849811 0.756184  ]]
[[0.1953556  0.04970691 0.75493747]]
[[0.19745645 0.05071387 0.7518297 ]]
[[0.19663988 0.05073077 0.75262934]]
[[0.19674769 0.05175915 0.75149316]]
[[0.19396985 0.05109764 0.7549325 ]]
[[0.19338645 0.05219807 0.7544154 ]]
[

# Directly Adding to Manual Dataset by Clicking

In [18]:
import cv2
import pickle
import numpy as np
import mediapipe as mp

Holistic = mp.solutions.holistic.Holistic

# Globals
counter = 0
current_frame = None
gesture = "scissors"
dataset_file = "data/"+gesture+"/Manual_dataset_RightHand_"+gesture+".pkl"

# Load existing dataset or create empty list
try:
    with open(dataset_file, "rb") as f:
        dataset = pickle.load(f)
except FileNotFoundError:
    dataset = []

def extract_landmarks(landmarks):
    # Access the .landmark attribute which is a list
    return np.array([[l.x, l.y] for l in landmarks.landmark], dtype=np.float32).flatten()

# Mouse callback
def mouse_click(event, x, y, flags, param):
    global counter, current_frame, dataset
    results = param  # param is right_hand_landmarks
    if event == cv2.EVENT_LBUTTONDOWN:
        if current_frame is not None:
            counter += 1
           
            print(counter)

            # Append landmarks if available
            if results is not None:
                ldk_temp = extract_landmarks(results)
                dataset.append(ldk_temp)
                # Save updated dataset
                with open(dataset_file, "wb") as f:
                    pickle.dump(dataset, f)
                print(f"Updated dataset with {len(ldk_temp)} landmarks.")

# Initialize webcam and Mediapipe
cap = cv2.VideoCapture(0)
cv2.namedWindow("Camera")

with Holistic(min_detection_confidence=0.5, min_tracking_confidence=0.5) as mp_holistic:
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        # frame = cv2.flip(frame, 1)
        current_frame = frame.copy()  # store for saving
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        results = mp_holistic.process(rgb_frame)
        # Pass right_hand_landmarks to mouse callback
        cv2.setMouseCallback("Camera", mouse_click, param=results.right_hand_landmarks)

        # Draw landmarks for visualization
        if results.right_hand_landmarks:
            mp.solutions.drawing_utils.draw_landmarks(
                frame,
                results.right_hand_landmarks,
                mp.solutions.holistic.HAND_CONNECTIONS
            )

        cv2.imshow("Camera", frame)
        if cv2.waitKey(1) & 0xFF == 27:  # ESC to quit
            break

cap.release()
cv2.destroyAllWindows()
